In [ ]:
%pip install datasets evaluate
%pip install transformers[torch]
%pip install accelerate
%pip install seqeval

In [2]:
from pathlib import Path
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np
import evaluate

In [3]:
encoder_id = "bert-base-cased"
# Use the fast tokenizer version, which supports word_ids()
tokenizer = BertTokenizerFast.from_pretrained(encoder_id)

In [4]:
def tokenize_and_align_labels(examples):
    # Use padding='max_length' to ensure consistent sequence lengths
    tokenized_inputs = tokenizer(
        examples["tokens"],
        padding="max_length",
        truncation=True,
        max_length=256,  # model_max_length from original hyperparameters
        is_split_into_words=True,
    )
    
    labels_aligned = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels_aligned.append(label_ids)
    
    tokenized_inputs["labels"] = labels_aligned
    return tokenized_inputs


# Load dataset and adjust columns
dataset_id = "DFKI-SLT/few-nerd"
dataset_name = "FewNERD"
dataset = load_dataset(dataset_id, "supervised")
dataset = dataset.remove_columns("ner_tags")
dataset = dataset.rename_column("fine_ner_tags", "ner_tags")
labels = dataset["train"].features["ner_tags"].feature.names

# Initialize model for token classification with BERT.
model = BertForTokenClassification.from_pretrained(encoder_id, num_labels=len(labels))

# Tokenize the datasets using the custom function.
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/37648 [00:00<?, ? examples/s]

In [7]:
# Prepare training arguments.
model_id = f"bert-token-classification-{encoder_id}-fewnerd-fine-super"
output_dir = Path("models") / model_id
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=64,  # Increased batch size for training
    per_device_eval_batch_size=64,   # Increased batch size for evaluation
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,  # Replace with fp16 if your hardware does not support bf16.
    logging_first_step=True,
    logging_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=3000,
    save_total_limit=2,
    dataloader_num_workers=2,
)

# Load evaluation metric.
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels_batch = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [labels[pred] for pred, label in zip(prediction, label_seq) if label != -100]
        for prediction, label_seq in zip(predictions, labels_batch)
    ]
    true_labels = [
        [labels[label] for pred, label in zip(prediction, label_seq) if label != -100]
        for prediction, label_seq in zip(predictions, labels_batch)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [8]:
# Use a dedicated data collator to handle padding of both inputs and labels.
data_collator = DataCollatorForTokenClassification(tokenizer)

# Initialize Trainer without directly passing tokenizer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model.
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
3000,0.208400,0.240257,0.664054,0.700086,0.681594,0.928858
6000,0.162900,0.240461,0.671152,0.708822,0.689473,0.930481


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=6177, training_loss=0.21210906536269963, metrics={'train_runtime': 1557.8189, 'train_samples_per_second': 253.753, 'train_steps_per_second': 3.965, 'total_flos': 5.167578462631373e+16, 'train_loss': 0.21210906536269963, 'epoch': 3.0})

In [9]:
# Evaluate on test set and save metrics.
test_metrics = trainer.evaluate(tokenized_datasets["test"], metric_key_prefix="test")
trainer.save_metrics("test", test_metrics)

# Save final model checkpoint.
trainer.save_model(output_dir / "checkpoint-final")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/home/student/miniconda3/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: building-other seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/student/miniconda3/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: building-hotel seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/student/miniconda3/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: location-GPE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/student/miniconda3/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: organization-showorganization seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/student/miniconda3/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: location-park seems not to be NE tag.
  warnin

## Saving and Loading the Model

In [10]:
# Save the entire model (including configuration and tokenizer state if desired)
trainer.save_model("weights.pth")


In [11]:
from transformers import BertForTokenClassification, BertTokenizerFast

# Load the saved model and tokenizer
model = BertForTokenClassification.from_pretrained("weights.pth")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")


## Making Predictions with Probabilities for Each Named Entity

In [20]:
import torch
import torch.nn.functional as F

# Sample text to predict on
sample_text = "John Doe visited New York City last week."

# Tokenize the sample text.
# Here, we split the text into words and use is_split_into_words=True to help with alignment.
words = sample_text.split()
inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256
)

# Forward pass through the model
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits  # Shape: (batch_size, seq_len, num_labels)

# Convert logits to probabilities using softmax
probs = F.softmax(logits, dim=-1)  # Shape: (batch_size, seq_len, num_labels)
predictions = torch.argmax(probs, dim=-1)  # Predicted label indices

# Get the mapping of tokens to words (word_ids)
word_ids = inputs.word_ids(batch_index=0)

# Initialize a dictionary to hold entity predictions and their top probabilities.
# We assume that tokens with a label other than "O" represent named entities.
results = {}

# Iterate over the token indices and group by word.
# Only consider the first token of a word (i.e. when word_ids change)
for idx, word_id in enumerate(word_ids):
    if word_id is None:
        continue
    # Check if this token is the first sub-token of a word.
    if idx == 0 or word_ids[idx] != word_ids[idx - 1]:
        # Get the probabilities for this token as a list
        token_probs = probs[0][idx].tolist()
        # Compute top 3 class probabilities (index, score)
        top3 = sorted(enumerate(token_probs), key=lambda x: x[1], reverse=True)[:3]
        # Map indices to label names and format the results.
        top3_labels = [(model.config.id2label[i], score) for i, score in top3]
        # If the predicted label is not the "O" (outside) label, store the entity.
        pred_label = model.config.id2label[predictions[0][idx].item()]
        if pred_label != "O":
            # Use the original word from the input list (note: words[word_id] gives a close approximation)
            results[words[word_id]] = top3_labels

print("Named Entity Predictions with Top 3 Probabilities:")
for entity, top_probs in results.items():
    print(f"Entity: {entity}")
    for label, prob in top_probs:
        print(f"  {label}: {prob:.4f}")


Named Entity Predictions with Top 3 Probabilities:
Entity: John
  LABEL_54: 0.6789
  LABEL_51: 0.1337
  LABEL_0: 0.1313
Entity: Doe
  LABEL_54: 0.6782
  LABEL_51: 0.1546
  LABEL_0: 0.1033
Entity: visited
  LABEL_0: 0.9999
  LABEL_54: 0.0000
  LABEL_21: 0.0000
Entity: New
  LABEL_21: 0.9937
  LABEL_0: 0.0033
  LABEL_25: 0.0009
Entity: York
  LABEL_21: 0.9943
  LABEL_0: 0.0032
  LABEL_25: 0.0007
Entity: City
  LABEL_21: 0.9081
  LABEL_0: 0.0877
  LABEL_25: 0.0011
Entity: last
  LABEL_0: 0.9999
  LABEL_21: 0.0000
  LABEL_25: 0.0000
Entity: week.
  LABEL_0: 1.0000
  LABEL_21: 0.0000
  LABEL_54: 0.0000


In [19]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizerFast, BertForTokenClassification

# Assume the model and tokenizer have been saved and then loaded as shown:
model_path = "weights.pth"  # Change this to your saved model path
model = BertForTokenClassification.from_pretrained(model_path)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

# Sample text for inference
sample_text = "John Doe visited New York City last week."

# Tokenize the full text with offsets. (Do NOT split manually to preserve original character offsets.)
encoded = tokenizer(
    sample_text,
    return_offsets_mapping=True,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt"
)
offset_mapping = encoded.pop("offset_mapping")  # shape: (1, seq_len, 2)

# Forward pass
with torch.no_grad():
    outputs = model(**encoded)
logits = outputs.logits  # shape: (1, seq_len, num_labels)
probs = F.softmax(logits, dim=-1)  # shape: (1, seq_len, num_labels)
predictions = torch.argmax(probs, dim=-1)  # shape: (1, seq_len)

# Now, group contiguous tokens with non-"O" predictions to form entities.
# We use the offset mapping to compute the character-level boundaries.
entities = []
current_entity = None

# Get the sequence length from the offsets.
seq_len = offset_mapping.size(1)

# Iterate over tokens (index i)
for i in range(seq_len):
    # Get the offset for token i.
    start_i, end_i = offset_mapping[0, i].tolist()
    # Skip special tokens or padding (offsets of (0, 0))
    if start_i == end_i == 0:
        continue

    label_id = predictions[0, i].item()
    # If the top predicted label is "O", then this token is not part of a named entity.
    if label_id == 0:
        # If we are in the middle of an entity, finish it.
        if current_entity is not None:
            entities.append(current_entity)
            current_entity = None
        continue

    # Otherwise, token is part of an entity.
    # Get the probability distribution for this token.
    token_probs = probs[0, i]  # tensor shape: (num_labels,)
    
    # If not currently grouping an entity, start one.
    if current_entity is None:
        current_entity = {
            "start": start_i,
            "end": end_i,
            "token_indices": [i],
            "prob_sum": token_probs.clone()  # accumulate probabilities (tensor)
        }
    else:
        # Check if token i is contiguous with the previous token in the entity.
        # We consider tokens contiguous if the current token’s start is at most 1 character after the previous token’s end.
        prev_end = current_entity["end"]
        if start_i - prev_end <= 1:
            # Extend the current entity.
            current_entity["end"] = end_i
            current_entity["token_indices"].append(i)
            current_entity["prob_sum"] += token_probs
        else:
            # Finish the current entity and start a new one.
            entities.append(current_entity)
            current_entity = {
                "start": start_i,
                "end": end_i,
                "token_indices": [i],
                "prob_sum": token_probs.clone()
            }

# Append any remaining entity.
if current_entity is not None:
    entities.append(current_entity)

# Process each entity: compute average probability across tokens, then get top-3 classes.
final_entities = []
for ent in entities:
    # Average probabilities over the tokens in this entity.
    avg_probs = ent["prob_sum"] / len(ent["token_indices"])
    avg_probs_list = avg_probs.tolist()
    # Get the top 3 (as (label_index, probability)) sorted by probability descending.
    top3 = sorted(enumerate(avg_probs_list), key=lambda x: x[1], reverse=True)[:3]
    
    # If the top prediction is "O" (i.e. label index 0), then ignore this entity.
    if top3[0][0] == 0:
        continue
    
    # Extract the surface form using character indices.
    surface = sample_text[ent["start"]:ent["end"]]
    
    # Build the result using label indices (as integers) and probabilities.
    entity_info = {
        "start": ent["start"],
        "end": ent["end"],
        "surface": surface,
        "top3": top3  # each entry is (label_index, probability)
    }
    final_entities.append(entity_info)

# Print the results.
print("Named Entity Predictions:")
for ent in final_entities:
    print(f"Entity: {ent['surface']}")
    print(f"  Begin index: {ent['start']}, End index: {ent['end']}")
    print("  Top 3 predictions:")
    for label_idx, prob in ent["top3"]:
        # Use the provided labels to show a human-readable label.
        label_name = labels[label_idx] if label_idx < len(labels) else f"LABEL_{label_idx}"
        print(f"    {label_idx} ({label_name}): {prob:.4f}")


Named Entity Predictions:
Entity: John Do
  Begin index: 0, End index: 7
  Top 3 predictions:
    54 (person-other): 0.6786
    51 (person-artist/author): 0.1441
    0 (O): 0.1173
Entity: New York City
  Begin index: 17, End index: 30
  Top 3 predictions:
    21 (location-GPE): 0.9654
    0 (O): 0.0314
    25 (location-other): 0.0009


In [23]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizerFast, BertForTokenClassification

# Load the model and tokenizer
model_path = "weights.pth"  # update with your path
model = BertForTokenClassification.from_pretrained(model_path)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")


# Sample text for inference
sample_text = "John Doe visited New York City last week."

# Tokenize the full text with offset mapping and word_ids
encoded = tokenizer(
    sample_text,
    return_offsets_mapping=True,
    return_tensors="pt",
    truncation=True,
    max_length=256
)
offset_mapping = encoded.pop("offset_mapping")  # shape: (1, seq_len, 2)
word_ids = encoded.word_ids(batch_index=0)         # list of word ids per token

# Forward pass
with torch.no_grad():
    outputs = model(**encoded)
logits = outputs.logits  # shape: (1, seq_len, num_labels)
probs = F.softmax(logits, dim=-1)  # shape: (1, seq_len, num_labels)
predictions = torch.argmax(probs, dim=-1)[0].tolist()  # shape: (seq_len,)

# Create a mapping from word index to aggregated information
word_info = {}
for i, word_id in enumerate(word_ids):
    if word_id is None:
        continue  # Skip special tokens
    # Get the token's offsets
    start, end = offset_mapping[0, i].tolist()
    # Initialize dictionary for the word if not seen
    if word_id not in word_info:
        word_info[word_id] = {
            "start": start,
            "end": end,
            "prob_sum": torch.zeros(probs.size(-1)),  # to accumulate probabilities
            "count": 0,
            "labels": []
        }
    else:
        # Update the end to the last sub-token's end
        word_info[word_id]["end"] = end
    # Accumulate the token's probabilities and count tokens
    word_info[word_id]["prob_sum"] += probs[0, i]
    word_info[word_id]["count"] += 1
    # Record predicted label for debugging if needed
    word_info[word_id]["labels"].append(predictions[i])

# Now, form entities by grouping contiguous words predicted as non-"O".
entities = []
current_entity = None
sorted_word_ids = sorted(word_info.keys())
for word_id in sorted_word_ids:
    info = word_info[word_id]
    # Calculate average probabilities for this word
    avg_probs = info["prob_sum"] / info["count"]
    avg_probs_list = avg_probs.tolist()
    # Get the top prediction for this word (we use average probabilities)
    top_pred = max(enumerate(avg_probs_list), key=lambda x: x[1])[0]
    
    # If the top prediction is "O", consider this word not part of an entity.
    if top_pred == 0:
        if current_entity is not None:
            entities.append(current_entity)
            current_entity = None
        continue

    # Start or extend an entity
    if current_entity is None:
        current_entity = {
            "start": info["start"],
            "end": info["end"],
            "words": [ (word_id, avg_probs) ]  # store tuple of word_id and its avg_probs tensor
        }
    else:
        # Check if the current word is contiguous in the original text
        # Here, contiguity is determined by the gap between current_entity end and current word start.
        if info["start"] - current_entity["end"] <= 1:
            current_entity["end"] = info["end"]
            current_entity["words"].append((word_id, avg_probs))
        else:
            entities.append(current_entity)
            current_entity = {
                "start": info["start"],
                "end": info["end"],
                "words": [(word_id, avg_probs)]
            }

if current_entity is not None:
    entities.append(current_entity)

# Process each entity to get the overall top 3 predictions.
final_entities = []
for ent in entities:
    # Average the probabilities over the words in the entity.
    summed_probs = torch.zeros(probs.size(-1))
    total_words = len(ent["words"])
    for _, word_avg_probs in ent["words"]:
        summed_probs += word_avg_probs
    entity_avg_probs = summed_probs / total_words
    entity_avg_probs_list = entity_avg_probs.tolist()
    top_probabilities_count = 3
    top_probabilites = sorted(enumerate(entity_avg_probs_list), key=lambda x: x[1], reverse=True)[:top_probabilities_count]
    
    # If the top prediction is "O", skip the entity.
    if top_probabilites[0][0] == 0:
        continue
    
    # Extract the surface form using character indices.
    surface = sample_text[ent["start"]:ent["end"]]
    
    final_entities.append({
        "start": ent["start"],
        "end": ent["end"],
        "surface": surface,
        "top_probabilites": top_probabilites  # each entry is (label_index, probability)
    })

# Print the results.
print("Named Entity Predictions:")
for ent in final_entities:
    print(f"Entity: {ent['surface']}")
    print(f"  Begin index: {ent['start']}, End index: {ent['end']}")
    print("  Top predictions:")
    for label_idx, prob in ent["top_probabilites"]:
        label_name = labels[label_idx] if label_idx < len(labels) else f"LABEL_{label_idx}"
        print(f"    {label_idx} ({label_name}): {prob:.4f}")


Named Entity Predictions:
Entity: John Doe
  Begin index: 0, End index: 8
  Top predictions:
    54 (person-other): 0.5881
    0 (O): 0.2302
    51 (person-artist/author): 0.1278
Entity: New York City
  Begin index: 17, End index: 30
  Top predictions:
    21 (location-GPE): 0.9654
    0 (O): 0.0314
    25 (location-other): 0.0009
